# Step 7: 评测方法论（downstream lm_eval + 性能/显存）

**目标**：建立完整的量化模型评测方法论——(1) **PPL**（s1-s5 在 quant env 已测，本节从产物读）；(2) **downstream 任务**（gsm8k/mmlu/ceval，用 `lm_eval --model vllm` 在本 vllm env 跑）；(3) **性能/显存**（vLLM `/metrics` 抓 KV-Cache 占用 + `vllm bench` 吞吐/延迟）。最终产出四维对比表（质量 | 显存 | 吞吐 | TTFT），作为 M3 的 finale。

**对应 OUTLINE 课时**：3.7 评测方法论（~55 分钟）。

> **双 env 提醒**：本 notebook 在 `steps/vllm` 子项目跑（vLLM + lm_eval env）。PPL 维的数据由 s1-s5 在 `steps/quant` env 产出、存共享 `out/`，本节**只读不重算**。


## 学完应能讲清（学完本节应能口头回答）
1. 量化模型评测为什么**至少要测 PPL + downstream 两类**？（PPL 是通用语言建模代理，downstream 是真实任务；OUTLINE 易错：PPL 几乎不变但 GSM8K 可能掉）
2. `lm_eval --model vllm` 的关键参数有哪些？为什么量化前后必须用**相同 seed/prompt/num_fewshot**？（否则对比不可信）
3. 测显存为什么推荐读 vLLM `/metrics` 的 `kv_cache_usage_perc`，而**差值法（nvidia-smi）会高估**？（nvidia-smi 显示进程已映射显存，含预分配非实际 KV）
4. `lm_eval[vllm]` extra 为什么不在 uv.lock、要手动 `uv pip install --python ./.venv/bin/python`？（与 vLLM wheel 自带 transformers 可能互锁）
5. 四维 Pareto 对比表是哪四维？为什么代码/数学任务对量化更**敏感（任务精度更易掉）**？提示：分别想清楚 (a) PPL 为什么测不出这类掉点（单步平均会把局部塌缩平滑掉）、(b) 数学任务为什么把局部塌缩放大成整体错（多步推理链上误差累积 + 罕见符号 token 被量化牺牲）。注意：吞吐维的「W4A16 提速/拖慢」是性能问题，与「精度敏感」是两回事。

In [ ]:
%%capture
import pathlib, os, json, subprocess, shlex
import ipytest
ipytest.autoconfig()
# lm_eval 可能未装（坑：lm_eval[vllm] extra 不在 uv.lock，需手动 uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'）
try:
    import lm_eval
    HAS_LM_EVAL = True
except ImportError:
    HAS_LM_EVAL = False
    print("[warn] lm_eval 未安装。先在 steps/vllm 跑：uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'")


In [ ]:
# Setup cell（双 env：本 notebook 在 steps/vllm 子项目跑；模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
# s7 跨两 env：PPL 数据从 steps/quant 产物读（out/ 共享）、downstream 在本 vllm env 跑 lm_eval。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
# quant 子项目产物目录（s1-s6 的 PPL/模型都在这）—— s7 读它做 PPL 维
QUANT_OUT        = OUT_ROOT  # 同一个 out/（模块根共享）
print("MODULE_ROOT =", MODULE_ROOT)
print("本 env:", "vllm" if pathlib.Path('.venv/bin/vllm').exists() else "?", "| lm_eval:", HAS_LM_EVAL)
print("OUT_ROOT =", OUT_ROOT, "| 读 quant 产物做 PPL 维对比")


## 原理：评测三维度（OUTLINE 3.7）

量化模型的「好」要**多维**测，单看任一维都会误判：

| 维度 | 工具 | 测什么 | env |
|---|---|---|---|
**质量-PPL** | transformers forward | 语言建模困惑度（通用代理，快） | quant（s1-s5 已测）|
**质量-downstream** | `lm_eval --model vllm` | gsm8k/mmlu/ceval 真实任务准确率 | vllm（本节）|
**性能** | `vllm bench serve/throughput/latency` | 吞吐 + TTFT + 延迟 | vllm |
**显存** | vLLM `/metrics`（Prometheus）| 权重 vs KV-Cache 占用 | vllm |

**关键易错点**（OUTLINE 3.7）：
- **PPL 几乎不变但 downstream 可能掉**：OUTLINE 3.8 明确——代码/数学任务对量化更敏感。所以**必须 downstream 实测**，PPL 不能当唯一判据。

  **为什么代码/数学任务对量化更敏感**（这里讲的是「任务精度更易掉」的因果，**不是**性能/吞吐维——吞吐维的「W4A16 在算力受限卡提速、H200 上 dequant 开销可能拖慢」是另一回事，别混淆）：
  - **PPL 是单步平均困惑度**：它对整个词表求一次 cross-entropy 再对所有位置取平均。少量 token 的局部塌缩会被海量的正常 token **平滑掉**——一个离群 token 拉高自己那一份 loss，但分子分母一起被平均稀释，整段文本的 PPL 几乎不动。
  - **代码/数学任务是多步符号推理链**：gsm8k 一道题要连续推导 5-20 步，**每一步的输出都是下一步的输入**。低 bit 权重误差在这样一条**长链**上会**累积放大**——第 3 步一个 token 算错，后面所有步骤都建立在错的前提上，整道题判错。
  - **罕见 token 的精确概率被牺牲**：量化（尤其 W4A16 / per-tensor）把高精度权重压成少数比特时，先牺牲的是**低频但关键**的符号 token（如 `)`、`=>`、数字、运算符）。这些 token 在 PPL 的平均里权重极小（出现次数少），但却是数学推导的承重墙——错一个整链崩。
  - **结论**：PPL 把这类「局部塌缩」平滑成几乎不变的全局平均，而 gsm8k 的「最终答案 exact_match」是 **0/1 裁决**、不容平滑。故 PPL≈不变、gsm8k 掉几个点是完全合理的。这正是为什么要下游实测、不能只看 PPL。

- **显存拆分要用 `/metrics` 不是 nvidia-smi**：`nvidia-smi` 显示进程已映射显存（含预分配），会**高估** KV-Cache。推荐读 `vllm:kv_cache_usage_perc` / `num_gpu_blocks`。差值法（空载记基线、峰值记高）有 nvidia-smi 偏差。
- **性能要用 server 端 `/metrics` + `vllm bench serve`**：V1 引擎下用 `llm.generate()` 拿 per-request TTFT **不可靠**。
- **lm_eval 必须固定 seed/prompt/num_fewshot**：量化前后不一致则对比无意义。

**`lm_eval[vllm]` 安装坑**（OUTLINE 附录 B）：extra 不进 uv.lock（与 vLLM wheel 自带 transformers 可能互锁），必须 `uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'`（`--python` 显式指定 venv，`uv pip` 不认项目 venv）。

### 端到端：评测在 M3 的位置（finale）

s1-s6 是调优过程，s7 是**裁决**——证明调优结果真有效。三维度（PPL/downstream/性能/显存）合成四维 Pareto 对比表，是整个 M3 的交付物，也是 OUTLINE 3.8 四维对比表的输入。同一套评测可套用到 FP8/AWQ/SmoothQuant 三方法对比。


## 亲手摸一摸：lm_eval 任务清单 + metrics 字段名

看 lm_eval 支持哪些任务、结果 dict 长什么样——为构造评测命令打底。


In [ ]:
## 摸一摸：lm_eval 任务 & 结果结构（不真跑，只看 API）
if HAS_LM_EVAL:
    from lm_eval.tasks import TaskManager
    tm = TaskManager()
    all_tasks = tm.all_tasks
    # 看课程关心的几个任务在不在
    wanted = ['gsm8k', 'mmlu', 'ceval-valid', 'cmmlu']
    present = [w for w in wanted if any(w in t for t in all_tasks)]
    print(f"lm_eval 任务总数 = {len(all_tasks)}")
    print(f"课程关心的任务命中: {present}")
    # 演示一个结果 dict 的字段（acc_norm / exact_match 等）
    print("\n典型 gsm8k 结果字段: results['gsm8k']['exact_match,none'] (取值 0-1)")
    print("典型 mmlu 结果字段:  results['mmlu']['acc,none'] (取值 0-1)")
else:
    print("lm_eval 未装，跳过摸一摸（先 uv pip install --python ./.venv/bin/python 'lm_eval[vllm]'）")


## 本步填空

1. **`run_downstream_eval(model_path, tasks, num_fewshot, limit)`**（判断型）—— 构造并（可选）执行 lm_eval 评测命令，返回结果。**为什么这么设计（填前先想）**：downstream 评测的参数选择（task、num_fewshot、limit）是判断决策——gsm8k 要 5-shot、mmlu 要 0/5-shot、limit 调优时小（250）最终大（全量）。本函数封装「选 task + 组参数 + 执行 + 抽指标」全流程，是 finale 的核心。
2. **`read_ppl_from_artifacts(out_root, method_tags)`** —— 从 quant env 产物（s1/s5 的 json + 各方法 config）读 PPL，合成 PPL 维对比。**为什么这么设计**：s7 不重算 PPL（quant env 已算），只读共享 `out/` 聚合——这是双 env 架构的精髓（PPL 留 quant、downstream 留 vllm，产物通过 out/ 交接）。


In [ ]:
def run_downstream_eval(model_path, tasks=('gsm8k',), num_fewshot=5, limit=250, execute=True):
    """判断型：构造 lm_eval 下游评测命令；execute=True 时真跑（需 GPU + lm_eval[vllm]），
    execute=False 只返回命令字符串（L1/L2 测命令构造逻辑）。
    返回 {'command': str, 'results': dict | None}。

    为什么这么设计（填前先想）：
    - task 选择是判断：gsm8k（数学）对量化最敏感、最能暴露掉点；mmlu/ceval 测通识。
    - num_fewshot：gsm8k 标配 5-shot（OUTLINE 3.5 命令）；limit=250 调优够、最终全量。
    - add_bos_token=True：Qwen2.5 评测惯例（OUTLINE 3.5 命令示例）。
    - 固定 seed（lm_eval 默认 random_seed=0）保证量化前后可比。
    命令模板（OUTLINE 3.5）：
      lm_eval --model vllm --model_args pretrained=<path>,add_bos_token=True \
        --tasks <tasks> --num_fewshot <n> --limit <limit>
    """
    # TODO: 1) tasks 是 tuple，join 成逗号串 'gsm8k,mmlu'。
    #       2) 构造命令：lm_eval --model vllm
    #          --model_args pretrained={model_path},add_bos_token=True
    #          --tasks {tasks_str} --num_fewshot {num_fewshot} --limit {limit}
    #          用 shlex.join 或空格拼接（注意 model_args 内逗号无空格）。
    #       3) 若 execute：subprocess.run(cmd, shell=True, capture_output=True, text=True) 跑；
    #          解析 stdout 找 'Results:' 后的 json，或调 lm_eval.simple_evaluate（更稳）。
    #          execute=False 时 results=None。
    #       4) 返回 {'command': cmd, 'results': results}。
    #   提示：L1/L2 测 execute=False（只验命令构造）；L3 execute=True。
    raise NotImplementedError


In [ ]:
def read_ppl_from_artifacts(out_root, method_tags):
    """从 quant env 产物（共享 out/）读各方法的 PPL，返回 {method: ppl}。
    读取顺序：优先 s5_pareto_curve.json（最准的调优后 PPL），
    否则 s1 产物，否则该方法目录无 PPL 记 None。

    为什么这么设计（填前先想）：s7 不重算 PPL——PPL 是 quant env（transformers forward）
    的活，s1/s5 已算好存 out/。s7 只聚合做对比维度。这是双 env 架构的交接点：
    quant env 产 PPL → out/ → vllm env 读 → 合成 finale 表。
    """
    # TODO: out_root 是 pathlib.Path，method_tags 是 ['fp8','awq','smoothquant'] 等。
    #       对每个 method：
    #       1) 找 out_root / f's5_pareto_curve_{method}.json' 或 's5_pareto_curve.json'，
    #          读 curve，取最后一个（全回退）或最优 k* 的 PPL。
    #       2) 否则找 out_root / f's1_*.json' 读 baseline。
    #       3) 都没有则 None。
    #       返回 {method: ppl_or_None}。
    raise NotImplementedError


In [ ]:
def test_run_downstream_eval_command_construction():
    # execute=False 只验命令构造（L1，不需 GPU/lm_eval）
    out = run_downstream_eval("/tmp/model", tasks=('gsm8k', 'mmlu'), num_fewshot=5, limit=250, execute=False)
    cmd = out['command']
    assert 'lm_eval --model vllm' in cmd
    assert 'pretrained=/tmp/model,add_bos_token=True' in cmd, "model_args 应含 path + add_bos_token"
    assert '--tasks gsm8k,mmlu' in cmd, "tasks 应逗号 join"
    assert '--num_fewshot 5' in cmd and '--limit 250' in cmd
    assert out['results'] is None, "execute=False 时 results 应为 None"

def test_run_downstream_eval_single_task():
    out = run_downstream_eval("/tmp/m", tasks=('gsm8k',), execute=False)
    assert '--tasks gsm8k' in out['command']

def test_read_ppl_from_artifacts_missing_returns_none(tmp_path):
    # 无任何产物 -> 全 None
    out = read_ppl_from_artifacts(tmp_path, ['fp8', 'awq'])
    assert out == {'fp8': None, 'awq': None}

def test_read_ppl_from_artifacts_reads_pareto(tmp_path):
    # 造一个 s5_pareto_curve.json
    json.dump({'curve': [(0, 12.0), (2, 9.0), (4, 7.5)]},
              open(tmp_path / 's5_pareto_curve.json', 'w'))
    out = read_ppl_from_artifacts(tmp_path, ['smoothquant'])
    assert out['smoothquant'] == 7.5, "应取曲线最优（最小）PPL"

# L1 必过守卫：ipytest.run 返回 pytest 退出码；非 0（有测试失败）→ 抛异常让 nbconvert 真挂。
# （用 ipytest.run() 而非 %%ipytest magic：magic 吞掉失败、exit_code 属性在本版不可靠。）
_ec = ipytest.run("-qq")
assert _ec == 0, f"L1 测试未全过（exit_code={_ec}），见上方 pytest 输出。"

## L2（CPU）：构造评测命令 + 读 PPL 聚合（不真跑 lm_eval）

L1/L2 验命令构造 + 产物读取逻辑（不需 GPU/lm_eval[vllm]）。造合成 quant 产物，验证 `read_ppl_from_artifacts` 正确聚合 PPL 维。


In [ ]:
## L2：合成 quant 产物，跑 PPL 维聚合
import tempfile
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    # 模拟 quant env 产的 PPL 曲线（三方法）
    json.dump({'curve': [(0,11.5),(2,9.0),(4,7.2)]}, open(td/'s5_pareto_curve_fp8.json','w'))
    json.dump({'curve': [(0,12.0),(2,9.5),(4,7.8)]}, open(td/'s5_pareto_curve_smoothquant.json','w'))
    # awq 无产物
    ppls = read_ppl_from_artifacts(td, ['fp8','awq','smoothquant'])
    print("PPL 维（从 quant 产物读）：")
    for m, p in ppls.items():
        print(f"  {m:15s} PPL={p}")
    assert ppls['fp8'] == 7.2 and ppls['smoothquant'] == 7.8
    assert ppls['awq'] is None, "无产物的方法应 None"

# 命令构造（不执行）
for method, path in [('FP8','/models/qwen-fp8'), ('AWQ','/models/qwen-awq'), ('SmoothQuant','/models/qwen-sq')]:
    out = run_downstream_eval(path, tasks=('gsm8k',), num_fewshot=5, limit=250, execute=False)
    print(f"\n{method} 下游评测命令：\n  {out['command']}")
print("\nL2 通过：命令构造 + PPL 维聚合逻辑正确（downstream 真跑见 L3）。")


## L3（H200，GPU 守卫）：真 7B 三方法四维评测 finale

在 7B 上对 FP8/AWQ/SmoothQuant 三方法跑 downstream（gsm8k 5-shot）+ 读 PPL 维 + 抓显存/吞吐，合成 OUTLINE 3.8 的四维 Pareto 对比表。这是 M3 的 finale 产物。

> L3 双守卫：除 GPU 外，reviewer 执行验证设 `SKIP_L3=1` 跳过（lm_eval 真跑慢）；真人/学员跑不设，L3 实证。


In [ ]:
import torch, os
def run_l3_four_dim_eval():
    # 三方法量化产物来源：M2 产 FP8/AWQ，或本模块 quant env 的 L3 步骤产 SmoothQuant/mixed-precision。
    # 设计 §5「复用 M2 产物或在 s7 重新量化」——这里**读 quant env 已产的模型目录**做 finale，
    # 不在 vllm env 重量化（llmcompressor 仅装在 quant env，vllm env 无该依赖；见双 env 提醒）。
    # 真人跑前先确保以下至少一个产物存在（跑对应 quant env L3 步骤即可生成）：
    #   FP8/AWQ          <- M2（models/Qwen2.5-7B-Instruct 的 FP8/AWQ 兄弟目录）
    #   SmoothQuant W8A8 <- quant env s3 L3（OUT_ROOT/s3_ignore 或 s3_all_quant）
    #   mixed-precision  <- quant env s4 L3（OUT_ROOT/s4_mixed_precision）
    #   W4A16 group=128  <- quant env s6 L3（OUT_ROOT/s6_group128）
    method_dirs = {
        'FP8':           MODEL_DIR.parent / 'qwen-fp8',          # M2 产物
        'AWQ':           MODEL_DIR.parent / 'qwen-awq',          # M2 产物
        'SmoothQuant':   OUT_ROOT / 's3_ignore',                 # quant env s3 L3 产物
    }
    available = {m: p for m, p in method_dirs.items() if p.exists()}
    missing = [m for m in method_dirs if m not in available]
    if missing:
        print(f"[warn] 以下方法产物不存在，将从四维表排除（先跑对应步骤生成）：{missing}")
        print("  FP8/AWQ: 跑 M2；SmoothQuant: 跑 quant env s3 的 L3（OUT_ROOT/s3_ignore）")
    if not available:
        print("[abort] 无任何方法产物可用——至少跑一个 quant env L3 步骤再回 s7。")
        return
    # 维1: PPL（从 quant 产物读；有则填，无则 None，downstream 仍可对比）
    ppls = read_ppl_from_artifacts(OUT_ROOT, list(available.keys()))
    # 维2: downstream gsm8k（每方法真跑 lm_eval[vllm]）
    table = {}
    for name, path in available.items():
        ds = run_downstream_eval(str(path), tasks=('gsm8k',), num_fewshot=5, limit=250, execute=True)
        gsm8k = None
        if ds['results']:
            try: gsm8k = ds['results']['gsm8k']['exact_match,none']
            except Exception: pass
        table[name] = {'ppl': ppls.get(name), 'gsm8k': gsm8k}
        print(f"  {name:13s} PPL={ppls.get(name)}  gsm8k={gsm8k}")
    # 维3/4: 显存/吞吐（需 vllm serve + /metrics，OUTLINE 3.7 命令）
    print("\n显存/吞吐维：需 vllm serve 后抓 /metrics（kv_cache_usage_perc）+ vllm bench serve")
    print("  curl -s http://localhost:8000/metrics | grep -E 'kv_cache_usage_perc|num_gpu_blocks'")
    json.dump(table, open(OUT_ROOT / 's7_four_dim_table.json', 'w'), indent=2)

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_four_dim_eval()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 命令构造+PPL 聚合逻辑）")

## 产物检查：四维 Pareto 对比表（finale）

合成 OUTLINE 3.8 四维对比表（质量 PPL+downstream | 显存 | 吞吐 | TTFT）。L3 跑完会在 `out/s7_four_dim_table.json` 产质量维；显存/吞吐维需 vLLM serve 后抓 `/metrics`（命令见 OUTLINE 3.7）。


In [ ]:
import json
p = OUT_ROOT / 's7_four_dim_table.json'
if p.exists():
    table = json.loads(p.read_text())
    print("=== 四维 Pareto 对比表 ===")
    print(f"{'方法':<14}{'PPL':>8}{'gsm8k':>10}")
    for name, d in table.items():
        ppl = f"{d['ppl']:.2f}" if d.get('ppl') is not None else 'N/A'
        gsm = f"{d['gsm8k']:.3f}" if d.get('gsm8k') is not None else 'N/A'
        print(f"{name:<14}{ppl:>8}{gsm:>10}")
    print("\n显存/吞吐维：见 vllm serve 的 /metrics（kv_cache_usage_perc）+ vllm bench serve")
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")
    print("\n[OUTLINE 3.7 抓显存命令示例]")
    print("  curl -s http://localhost:8000/metrics | grep -E 'kv_cache_usage_perc|num_gpu_blocks'")
